In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab data access)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Lightweight LightGBM 3-Way Split (Train/Val/Test) Stacking Pipeline (`models/train_oof_logistic_regression_stacking.ipynb`)

This notebook trains the **ultra-lightweight Stacking Pipeline natively in Python using scikit-learn `LGBMClassifier` & `LogisticRegression`** for `m2cgen` C transpilation:

### 3-Way Stratified Data Structure (`Train / Validation / Test`)
- Strictly drops any observation with $\ge 1$ null / NA value in the 15 main raw features.
- Splits data into **Train (98%)**, **Validation (1%)**, and **Holdout Test (1%)** based on `config/triage_conf.json`.
- Base LightGBM sub-models (`l1_prod`..`l3b_prod`) are trained on the **Train set (98%)** with early stopping evaluated on the **Validation set (1%)**.
- The Multinomial Logistic Regression meta-learner and **balanced class weighting** are calibrated with **5-Fold Cross-Validation on the Validation set (1%)**.
- Evaluates production pipeline on the **Holdout Test set (1%)** with Confusion Matrix, Density Distribution, and ROC-AUC curves.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Raw Dataset, Filter Complete Cases & Stratified 3-Way Split (Train/Val/Test)
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)

set.seed(config$training$random_state)

stratified_sample <- function(y, fraction, seed = 42) {
  set.seed(seed)
  idx_list <- split(seq_along(y), y)
  sampled <- unlist(lapply(idx_list, function(idx) {
    n_sample <- max(1, round(length(idx) * fraction))
    sample(idx, size = n_sample)
  }))
  return(sort(sampled))
}

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
target_col_name <- config$classes$target_col

initial_total_rows <- nrow(raw_df)
cat("========================================================================\n")
cat(sprintf("  INITIAL DATASET LOADED: %d Total Rows, %d Total Columns\n", initial_total_rows, ncol(raw_df)))
cat("========================================================================\n")
if (target_col_name %in% names(raw_df)) {
  cat("Initial ESI Target Distribution (including NAs):\n")
  print(table(raw_df[[target_col_name]], useNA = "ifany"))
  cat("------------------------------------------------------------------------\n")
}

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) {
    return(raw_df[[col_name]])
  } else {
    return(rep(NA, nrow(raw_df)))
  }
}

raw_esi <- as.character(raw_df[[target_col_name]])

df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  pulse_min               = get_vec("pulse_min"),
  resp_min                = get_vec("resp_min"),
  spo2_min                = get_vec("spo2_min"),
  sbp_min                 = get_vec("sbp_min"),
  pulse_max               = get_vec("pulse_max"),
  resp_max                = get_vec("resp_max"),
  spo2_max                = get_vec("spo2_max"),
  sbp_max                 = get_vec("sbp_max"),
  target_col              = factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
)

# Strictly drop any row containing at least 1 null/NA value across all 15 raw features or target
df_master <- na.omit(df_master)
df_master$target_num <- as.numeric(as.character(df_master$target_col))

clean_total_rows <- nrow(df_master)
dropped_rows     <- initial_total_rows - clean_total_rows

cat(sprintf("Missing Values Filter: Dropped %d rows with >= 1 NA feature (Retained %d Complete Cases, %.2f%%)\n", 
            dropped_rows, clean_total_rows, (clean_total_rows / initial_total_rows) * 100))
cat("Cleaned ESI Distribution (100% complete cases):\n")
print(table(df_master$target_col))
cat("------------------------------------------------------------------------\n")

# Stratified 3-Way Partitioning based on triage_conf.json
test_size <- config$training$test_size
val_size  <- config$training$val_size
seed_val  <- config$training$random_state

# 1. Extract Stratified Holdout Test Set (e.g., 1%)
idx_test <- stratified_sample(df_master$target_col, test_size, seed = seed_val)
test_df_clean  <- df_master[idx_test, ]
rem_df         <- df_master[-idx_test, ]

# 2. Extract Stratified Validation Set from remainder (e.g., 1% of total)
val_adj_fraction <- val_size / (1 - test_size)
idx_val <- stratified_sample(rem_df$target_col, val_adj_fraction, seed = seed_val + 1)
val_df_clean   <- rem_df[idx_val, ]
train_df_clean <- rem_df[-idx_val, ]

train_mat_export <- as.matrix(cbind(train_df_clean[, 1:15], target = train_df_clean$target_num))
val_mat_export   <- as.matrix(cbind(val_df_clean[, 1:15],   target = val_df_clean$target_num))
test_mat_export  <- as.matrix(cbind(test_df_clean[, 1:15],  target = test_df_clean$target_num))

cat(sprintf("3-Way Partition Complete:\n  Train Set      = %d rows (%.2f%%)\n  Validation Set = %d rows (%.2f%%)\n  Holdout Test   = %d rows (%.2f%%)\n", 
            nrow(train_mat_export), (nrow(train_mat_export) / clean_total_rows) * 100,
            nrow(val_mat_export),   (nrow(val_mat_export) / clean_total_rows) * 100,
            nrow(test_mat_export),  (nrow(test_mat_export) / clean_total_rows) * 100))
cat("========================================================================\n")

In [ ]:
# ---------------------------------------------------------
# Step 2: Feature Engineering, Base Sub-Models Training & Validation Meta-Learner (5-Fold CV)
# ---------------------------------------------------------
import os
import pickle
import numpy as np
import pandas as pd
from rpy2.robjects import r
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss
import lightgbm as lgb

train_mat_in = np.array(r('train_mat_export'), dtype=np.float64)
val_mat_in   = np.array(r('val_mat_export'),   dtype=np.float64)
test_mat_in  = np.array(r('test_mat_export'),  dtype=np.float64)

raw_mat_tr  = train_mat_in[:, :15]
y_train     = train_mat_in[:, 15].astype(int)

raw_mat_val = val_mat_in[:, :15]
y_val       = val_mat_in[:, 15].astype(int)

raw_mat_ts  = test_mat_in[:, :15]
y_test      = test_mat_in[:, 15].astype(int)

def build_38_feature_matrix(raw_mat):
    N = len(raw_mat)
    X = np.zeros((N, 38), dtype=np.float64)
    X[:, :15] = raw_mat
    
    t_hr = raw_mat[:, 3]; t_sbp = raw_mat[:, 4]; t_rr = raw_mat[:, 5]; t_o2 = raw_mat[:, 6]
    pulse_min = raw_mat[:, 7]; resp_min = raw_mat[:, 8]; spo2_min = raw_mat[:, 9]; sbp_min = raw_mat[:, 10]
    pulse_max = raw_mat[:, 11]; resp_max = raw_mat[:, 12]; spo2_max = raw_mat[:, 13]; sbp_max = raw_mat[:, 14]
    
    hr_rng   = pulse_max - pulse_min
    rr_rng   = resp_max - resp_min
    spo2_rng = spo2_max - spo2_min
    sbp_rng  = sbp_max - sbp_min
    
    X[:, 15] = (t_o2 < 90).astype(float)
    X[:, 16] = ((t_o2 > 90) & (t_o2 < 94)).astype(float)
    X[:, 17] = (t_rr < 10).astype(float)
    X[:, 18] = (t_rr > 30).astype(float)
    X[:, 19] = (t_sbp <= 90).astype(float)
    X[:, 20] = (t_sbp > 220).astype(float)
    X[:, 21] = (t_hr < 40).astype(float)
    X[:, 22] = ((t_hr > 40) & (t_hr < 60)).astype(float)
    X[:, 23] = (t_hr > 150).astype(float)
    X[:, 24] = ((t_hr > 100) & (t_hr < 150)).astype(float)
    X[:, 25] = hr_rng; X[:, 26] = rr_rng; X[:, 27] = spo2_rng; X[:, 28] = sbp_rng
    X[:, 29] = t_hr / np.where(t_sbp == 0, 1.0, t_sbp)
    X[:, 30] = t_hr - hr_rng
    X[:, 31] = t_sbp - sbp_rng
    X[:, 32] = t_rr - rr_rng
    X[:, 33] = t_o2 - spo2_rng
    X[:, 34] = t_o2 / np.where(t_rr == 0, 1.0, t_rr)
    X[:, 35] = spo2_rng / np.where(spo2_max == 0, 1.0, spo2_max)
    X[:, 36] = hr_rng / (t_hr + 1.0)
    X[:, 37] = (t_rr / np.where(t_o2 == 0, 1.0, t_o2)) * 100.0
    return X

X_train_raw = build_38_feature_matrix(raw_mat_tr)
X_val_raw   = build_38_feature_matrix(raw_mat_val)
X_test_raw  = build_38_feature_matrix(raw_mat_ts)

feature_names = [
    'age', 'cc_breathingdifficulty', 'gender', 'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_rr', 'triage_vital_o2',
    'pulse_min', 'resp_min', 'spo2_min', 'sbp_min', 'pulse_max', 'resp_max', 'spo2_max', 'sbp_max',
    'is_dyspnea_total', 'is_dyspnea_moderate', 'is_bradypnea', 'is_tachypnea', 'is_hypotension', 'is_hypertension',
    'is_bradycardia_total', 'is_bradycardia_moderate', 'is_tachycardia_total', 'is_tachycardia_moderate',
    'hr_range', 'rr_range', 'spo2_range', 'sbp_range',
    'shock_index', 'hr_mid_to_triage', 'sbp_mid_to_triage', 'rr_mid_to_triage', 'spo2_mid_to_triage',
    'rox_index', 'spo2_drop_ratio', 'hr_instability_ratio', 'bif'
]

cont_cols_idx = [0, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37]

scaler = StandardScaler()
X_train = X_train_raw.copy()
X_val   = X_val_raw.copy()
X_test  = X_test_raw.copy()

X_train[:, cont_cols_idx] = scaler.fit_transform(X_train_raw[:, cont_cols_idx])
X_val[:, cont_cols_idx]   = scaler.transform(X_val_raw[:, cont_cols_idx])
X_test[:, cont_cols_idx]  = scaler.transform(X_test_raw[:, cont_cols_idx])

def numpy_smote(X, y_bin, seed=42):
    np.random.seed(seed)
    pos_mask = (y_bin == 1)
    neg_mask = (y_bin == 0)
    n_pos = np.sum(pos_mask)
    n_neg = np.sum(neg_mask)
    if n_pos == 0 or n_neg == 0 or n_pos == n_neg:
        return X, y_bin
    if n_pos < n_neg:
        min_X = X[pos_mask]; target_syn = n_neg - n_pos; min_label = 1
    else:
        min_X = X[neg_mask]; target_syn = n_pos - n_neg; min_label = 0
    n_min = len(min_X)
    syn_X = np.zeros((target_syn, X.shape[1]))
    for i in range(target_syn):
        idx1 = np.random.randint(0, n_min)
        idx2 = np.random.randint(0, n_min)
        alpha = np.random.rand()
        syn_X[i] = min_X[idx1] + alpha * (min_X[idx2] - min_X[idx1])
    return np.vstack([X, syn_X]), np.hstack([y_bin, np.full(target_syn, min_label)])

lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbosity': -1,
    'random_state': 42
}

# ---------------------------------------------------------
# Train Base Sub-Models on Train Set (98%) with Early Stopping on Validation Set (1%)
# ---------------------------------------------------------
print("Training Base Sub-Models on Train Set (98%) with Validation Set Early Stopping...")

# Layer 1: ESI 1 vs (ESI 2..5)
X_sm1, y_sm1 = numpy_smote(X_train, (y_train == 1).astype(int))
l1_prod = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l1_prod.fit(X_sm1, y_sm1, eval_set=[(X_val, (y_val == 1).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# Layer 2: ESI 2,3 vs ESI 4,5 (trained on non-ESI 1)
m2_tr  = (y_train != 1)
m2_val = (y_val != 1)
X_sm2, y_sm2 = numpy_smote(X_train[m2_tr], np.isin(y_train[m2_tr], [2, 3]).astype(int))
l2_prod = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l2_prod.fit(X_sm2, y_sm2, eval_set=[(X_val[m2_val], np.isin(y_val[m2_val], [2, 3]).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# Layer 3A: ESI 2 vs ESI 3
m3a_tr  = np.isin(y_train, [2, 3])
m3a_val = np.isin(y_val, [2, 3])
X_sm3a, y_sm3a = numpy_smote(X_train[m3a_tr], (y_train[m3a_tr] == 2).astype(int))
l3a_prod = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l3a_prod.fit(X_sm3a, y_sm3a, eval_set=[(X_val[m3a_val], (y_val[m3a_val] == 2).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# Layer 3B: ESI 4 vs ESI 5
m3b_tr  = np.isin(y_train, [4, 5])
m3b_val = np.isin(y_val, [4, 5])
X_sm3b, y_sm3b = numpy_smote(X_train[m3b_tr], (y_train[m3b_tr] == 4).astype(int))
l3b_prod = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l3b_prod.fit(X_sm3b, y_sm3b, eval_set=[(X_val[m3b_val], np.isin(y_val[m3b_val], [4, 5]).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# ---------------------------------------------------------
# Generate Base Probabilities for Validation Set (1%)
# ---------------------------------------------------------
p1_val  = l1_prod.predict_proba(X_val)[:, 1]
p2_val  = l2_prod.predict_proba(X_val)[:, 1]
p3a_val = l3a_prod.predict_proba(X_val)[:, 1]
p3b_val = l3b_prod.predict_proba(X_val)[:, 1]

val_probs = np.zeros((len(X_val), 5))
val_probs[:, 0] = p1_val
val_probs[:, 1] = (1 - p1_val) * p2_val * p3a_val
val_probs[:, 2] = (1 - p1_val) * p2_val * (1 - p3a_val)
val_probs[:, 3] = (1 - p1_val) * (1 - p2_val) * p3b_val
val_probs[:, 4] = (1 - p1_val) * (1 - p2_val) * (1 - p3b_val)

# ---------------------------------------------------------
# 5-Fold Cross-Validation & Balanced Class Weighting on Validation Set
# ---------------------------------------------------------
print("Calibrating Meta-Learner with 5-Fold Cross-Validation on the Validation Set...")
skf_val = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for fold, (v_tr_idx, v_val_idx) in enumerate(skf_val.split(val_probs, y_val)):
    cv_lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    cv_lr.fit(val_probs[v_tr_idx], y_val[v_tr_idx])
    pred_v = cv_lr.predict(val_probs[v_val_idx])
    acc_v  = np.mean(pred_v == y_val[v_val_idx])
    cv_scores.append(acc_v)

print(f"Validation 5-Fold CV Meta-Learner Accuracy: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})")

# Fit Final Meta-Learner with Balanced Class Weighting on Entire Validation Set
meta_logreg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
meta_logreg.fit(val_probs, y_val)

# ---------------------------------------------------------
# Evaluate on Holdout Test Set (1%)
# ---------------------------------------------------------
p1_test  = l1_prod.predict_proba(X_test)[:, 1]
p2_test  = l2_prod.predict_proba(X_test)[:, 1]
p3a_test = l3a_prod.predict_proba(X_test)[:, 1]
p3b_test = l3b_prod.predict_proba(X_test)[:, 1]

test_probs_prod = np.zeros((len(X_test), 5))
test_probs_prod[:, 0] = p1_test
test_probs_prod[:, 1] = (1 - p1_test) * p2_test * p3a_test
test_probs_prod[:, 2] = (1 - p1_test) * p2_test * (1 - p3a_test)
test_probs_prod[:, 3] = (1 - p1_test) * (1 - p2_test) * p3b_test
test_probs_prod[:, 4] = (1 - p1_test) * (1 - p2_test) * (1 - p3b_test)

# Export Bundle to deploy/py_oof_stacking_bundle.pkl
deploy_dir = '../deploy' if os.path.exists('../deploy') else 'deploy'
os.makedirs(deploy_dir, exist_ok=True)

bundle_data = {
    'l1_prod': l1_prod,
    'l2_prod': l2_prod,
    'l3a_prod': l3a_prod,
    'l3b_prod': l3b_prod,
    'meta_logreg': meta_logreg,
    'scaler_means': scaler.mean_,
    'scaler_sds': scaler.scale_,
    'cont_cols_idx': cont_cols_idx,
    'feature_names': feature_names
}

with open(os.path.join(deploy_dir, 'py_oof_stacking_bundle.pkl'), 'wb') as f:
    pickle.dump(bundle_data, f)

print(f"Python Native Calibrated LightGBM Bundle saved to: {os.path.join(deploy_dir, 'py_oof_stacking_bundle.pkl')}")

In [ ]:
# ---------------------------------------------------------
# Step 3: Holdout Test Set Evaluation & Detailed Per-Class Breakdown
# ---------------------------------------------------------
preds_meta_logreg = meta_logreg.predict(test_probs_prod)
probs_meta_logreg = meta_logreg.predict_proba(test_probs_prod)

def get_per_class_breakdown(y_true, y_pred, probs, pipeline_name):
    classes = [1, 2, 3, 4, 5]
    rows = []
    recalls, specs, bal_accs, aucs = [], [], [], []
    for idx, cls in enumerate(classes):
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal  = (rec + spec) / 2.0
        try: auc = roc_auc_score(y_bin_true, probs[:, idx])
        except Exception: auc = 0.0
        recalls.append(rec); specs.append(spec); bal_accs.append(bal); aucs.append(auc)
        rows.append({
            'Pipeline': pipeline_name,
            'Class': f'ESI_{cls}',
            'Recall': round(rec, 4),
            'Specificity': round(spec, 4),
            'Balanced_Accuracy': round(bal, 4),
            'ROC_AUC': round(auc, 4)
        })
    rows.append({
        'Pipeline': pipeline_name,
        'Class': 'Macro_Average',
        'Recall': round(np.mean(recalls), 4),
        'Specificity': round(np.mean(specs), 4),
        'Balanced_Accuracy': round(np.mean(bal_accs), 4),
        'ROC_AUC': round(np.mean(aucs), 4)
    })
    return pd.DataFrame(rows)

report_df = get_per_class_breakdown(y_test, preds_meta_logreg, probs_meta_logreg, 'OOF_Stacking_Multinomial_Logistic_Regression')

print("========================================================================================")
print("   HOLDOUT TEST SET REPORT: PYTHON NATIVE OOF MULTINOMIAL LOGISTIC META-LEARNER")
print("========================================================================================")
print(report_df.to_string(index=False))
print("========================================================================================\n")

reports_dir = '../reports' if os.path.exists('../reports') else 'reports'
os.makedirs(reports_dir, exist_ok=True)

report_df.to_csv(os.path.join(reports_dir, 'oof_multinomial_logistic_stacking_report.csv'), index=False)
print(f"Report saved to {os.path.join(reports_dir, 'oof_multinomial_logistic_stacking_report.csv')}")

In [ ]:
# ---------------------------------------------------------
# Step 4: Confusion Matrix Graph for Holdout Test Benchmark
# ---------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

plots_dir = '../plots' if os.path.exists('../plots') else 'plots'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

esi_labels = [f"ESI {i}" for i in range(1, 6)]

# Compute confusion matrix
cm_meta      = confusion_matrix(y_test, preds_meta_logreg, labels=[1, 2, 3, 4, 5])
cm_meta_norm = cm_meta.astype('float') / cm_meta.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(9, 7))

annot_meta = np.empty_like(cm_meta, dtype=object)
for i in range(5):
    for j in range(5):
        annot_meta[i, j] = f"{cm_meta[i, j]}\n({cm_meta_norm[i, j]*100:.1f}%)"

sns.heatmap(cm_meta_norm, annot=annot_meta, fmt='', cmap='Greens', cbar=True,
            xticklabels=esi_labels, yticklabels=esi_labels, ax=ax, vmin=0, vmax=1)
ax.set_title('Holdout Test Confusion Matrix', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Predicted ESI Level', fontsize=11, fontweight='bold')
ax.set_ylabel('True ESI Level', fontsize=11, fontweight='bold')

plt.tight_layout()

cm_path = os.path.join(plots_dir, 'holdout_test_confusion_matrix.png')
plt.savefig(cm_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'holdout_test_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Confusion Matrix Graph saved to: {cm_path}")

In [ ]:
# ---------------------------------------------------------
# Step 5: Density Data Distribution Graphs (Saved to Individual PNG per Feature)
# ---------------------------------------------------------
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

plots_dir = '../plots' if os.path.exists('../plots') else 'plots'
density_dir = os.path.join(plots_dir, 'density')
density_img_dir = os.path.join(plots_dir, 'image', 'density')
os.makedirs(density_dir, exist_ok=True)
os.makedirs(density_img_dir, exist_ok=True)

raw_feature_names_15 = [
    'age', 'cc_breathingdifficulty', 'gender', 'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_rr', 'triage_vital_o2',
    'pulse_min', 'resp_min', 'spo2_min', 'sbp_min', 'pulse_max', 'resp_max', 'spo2_max', 'sbp_max'
]

# Construct full DataFrame of raw features and true ESI labels
df_raw_all = pd.DataFrame(raw_mat_tr, columns=raw_feature_names_15)
df_raw_all['ESI'] = [f"ESI {k}" for k in y_train]

esi_palette = {
    'ESI 1': '#d62728',  # Red (Resuscitation)
    'ESI 2': '#ff7f0e',  # Orange (Emergent)
    'ESI 3': '#2ca02c',  # Green (Urgent)
    'ESI 4': '#1f77b4',  # Blue (Less Urgent)
    'ESI 5': '#9467bd'   # Purple (Non-urgent)
}

print(f"Saving individual feature density distribution plots to: {density_dir}")

for feat in raw_feature_names_15:
    fig, ax = plt.subplots(figsize=(8, 5))
    
    if feat == 'gender':
        gen_prop = df_raw_all.groupby('ESI')['gender'].mean().reset_index(name='Male_Proportion')
        sns.barplot(data=gen_prop, x='ESI', y='Male_Proportion', palette=esi_palette, ax=ax, edgecolor='black')
        ax.set_title("gender (Male Proportion by ESI Level)", fontsize=13, fontweight='bold', pad=12)
        ax.set_xlabel("ESI Level", fontsize=11, fontweight='bold')
        ax.set_ylabel("Proportion (1=Male)", fontsize=11, fontweight='bold')
        ax.set_ylim(0, 1.0)
        for p in ax.patches:
            ax.annotate(f"{p.get_height()*100:.1f}%",
                        (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=10, fontweight='bold', xytext=(0, 2),
                        textcoords='offset points')
    elif feat == 'cc_breathingdifficulty':
        cc_prop = df_raw_all.groupby('ESI')['cc_breathingdifficulty'].mean().reset_index(name='Proportion')
        sns.barplot(data=cc_prop, x='ESI', y='Proportion', palette=esi_palette, ax=ax, edgecolor='black')
        ax.set_title("cc_breathingdifficulty (Prevalence by ESI Level)", fontsize=13, fontweight='bold', pad=12)
        ax.set_xlabel("ESI Level", fontsize=11, fontweight='bold')
        ax.set_ylabel("Prevalence (1=Yes)", fontsize=11, fontweight='bold')
        ax.set_ylim(0, max(0.4, cc_prop['Proportion'].max() * 1.25))
        for p in ax.patches:
            ax.annotate(f"{p.get_height()*100:.1f}%",
                        (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=10, fontweight='bold', xytext=(0, 2),
                        textcoords='offset points')
    else:
        sns.kdeplot(
            data=df_raw_all,
            x=feat,
            hue='ESI',
            palette=esi_palette,
            common_norm=False,
            fill=True,
            alpha=0.20,
            linewidth=2.0,
            ax=ax
        )
        ax.set_title(f"Raw Feature Density Distribution: {feat}", fontsize=13, fontweight='bold', pad=12)
        ax.set_xlabel(feat, fontsize=11, fontweight='bold')
        ax.set_ylabel("Density", fontsize=11, fontweight='bold')
    
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    
    out_file = f"density_{feat}.png"
    plt.savefig(os.path.join(density_dir, out_file), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(density_img_dir, out_file), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved: {os.path.join(density_dir, out_file)}")

print(f"All 15 individual density plots successfully saved in {density_dir}")

In [ ]:
# ---------------------------------------------------------
# Step 6: Multiclass ROC-AUC Curve Analysis (Holdout Test Benchmark)
# ---------------------------------------------------------
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

plots_dir = '../plots' if os.path.exists('../plots') else 'plots'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

# Binarize the true test labels for multiclass ROC computation (1 to 5)
classes = [1, 2, 3, 4, 5]
y_test_bin = label_binarize(y_test, classes=classes)
n_classes  = len(classes)

# Compute ROC curve and ROC area for each class
fpr = dict()
tpr = dict()
roc_auc = dict()

for i, cls in enumerate(classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], probs_meta_logreg[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Compute micro-average ROC curve and ROC area
fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), probs_meta_logreg.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# Compute macro-average ROC curve and ROC area
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= n_classes

fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

# Plot ROC Curves
plt.figure(figsize=(9, 8))

esi_colors = {
    0: '#d62728',  # ESI 1: Red
    1: '#ff7f0e',  # ESI 2: Orange
    2: '#2ca02c',  # ESI 3: Green
    3: '#1f77b4',  # ESI 4: Blue
    4: '#9467bd'   # ESI 5: Purple
}

# Plot micro and macro average ROC curves
plt.plot(fpr["micro"], tpr["micro"],
         label=f"Micro-Average (AUC = {roc_auc['micro']:.4f})",
         color='#e377c2', linestyle=':', linewidth=2.5)

plt.plot(fpr["macro"], tpr["macro"],
         label=f"Macro-Average (AUC = {roc_auc['macro']:.4f})",
         color='#17becf', linestyle='--', linewidth=2.5)

# Plot individual class ROC curves
for i, cls in enumerate(classes):
    plt.plot(fpr[i], tpr[i], color=esi_colors[i], linewidth=2.0,
             label=f"ESI {cls} (AUC = {roc_auc[i]:.4f})")

# Diagonal random baseline
plt.plot([0, 1], [0, 1], 'k--', color='gray', linewidth=1.2, label='Random Guess (AUC = 0.5000)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=12, fontweight='bold')
plt.title('Holdout Test ROC-AUC Curves (One-vs-Rest by ESI Level)', fontsize=14, fontweight='bold', pad=12)
plt.legend(loc="lower right", fontsize=10.5, frameon=True, framealpha=0.95)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

roc_plot_path = os.path.join(plots_dir, 'holdout_test_roc_auc_curve.png')
plt.savefig(roc_plot_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'holdout_test_roc_auc_curve.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"ROC-AUC Curve Graph saved to: {roc_plot_path}")